## Context

The 2008 Global Financial Crisis illustrates how multiple risk systems—credit, market, liquidity, and counterparty—can fail simultaneously.

Financial institutions relied on:

- historical default data  
- credit ratings  
- correlation assumptions  

These inputs proved unreliable when housing prices declined and systemic stress increased.

---

## Why This Case Matters

This case extends beyond a single failure mechanism.

It demonstrates how:

- market-implied signals (CDS spreads) diverge from historical models  
- credit metrics (Altman Z) reveal underlying fragility  
- correlation and wrong-way risk amplify losses  
- collateral and downgrade triggers convert credit events into liquidity crises  

---

## Key Insight

> Systemic crises emerge when multiple risk models fail simultaneously and reinforce each other.

## Analytical Lens

**Insight:** Risk models fail when interdependencies across credit, market, and liquidity systems are underestimated.

**Quant Takeaway:** Market-implied default probabilities, structural credit metrics, and portfolio loss models produce materially different risk estimates under stress, indicating model divergence.

**Model Implication:** Risk management must integrate multiple models (CDS, Z-score, VaR, CVA) and explicitly account for interaction effects such as correlation, liquidity constraints, and wrong-way risk.

In [1]:

# Setup: imports and display options
import math
import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option('display.float_format', lambda x: f"{x:,.4f}")
print("Environment ready")


Environment ready



## Data Tables & Assumptions
These match the in-class case but are preloaded for your individual work.


In [2]:

# Table 1: CDS spreads & recovery (bps, decimal)
cds_data = pd.DataFrame({
    'Tranche': ['BBB', 'AAA'],
    'CDS_Spread_bps': [240, 30],
    'Recovery_rate': [0.40, 0.40]
})

# Table 2: Mortgage Originator Financials (USD millions)
originator = pd.DataFrame({
    'Metric': ['Working Capital','Retained Earnings','EBIT','Market Value of Equity',
               'Book Value of Liabilities','Sales','Total Assets'],
    'Value_M': [50, 20, 15, 80, 120, 200, 250]
})

# Table 3: Ratings Transition (1-year, BBB row only)
transition = pd.DataFrame({
    'To': ['AAA','AA','A','BBB','BB','B','CCC/C','D'],
    'Prob_%': [0.59, 1.05, 1.70, 1.88, 4.20, 8.20, 25.00, 50.00]
})

# Assumptions
discount_factor_annual = 0.95
confidence_level = 0.999
rho = 0.15

print(cds_data)
print(originator)
print(transition)


  Tranche  CDS_Spread_bps  Recovery_rate
0     BBB             240         0.4000
1     AAA              30         0.4000
                      Metric  Value_M
0            Working Capital       50
1          Retained Earnings       20
2                       EBIT       15
3     Market Value of Equity       80
4  Book Value of Liabilities      120
5                      Sales      200
6               Total Assets      250
      To  Prob_%
0    AAA  0.5900
1     AA  1.0500
2      A  1.7000
3    BBB  1.8800
4     BB  4.2000
5      B  8.2000
6  CCC/C 25.0000
7      D 50.0000


In [3]:

# --- Helper Functions ---

def hazard_rate(spread_bps: float, recovery: float) -> float:
    # Risk-neutral average annual default probability (hazard rate) ~ s / (1 - R).
    s = spread_bps / 10000.0
    return s / (1 - recovery)


def altman_z(financials: dict) -> float:
    # Altman Z-Score for manufacturing firms (classic 5-factor version).
    X1 = financials['WC'] / financials['TA']
    X2 = financials['RE'] / financials['TA']
    X3 = financials['EBIT'] / financials['TA']
    X4 = financials['MVE'] / financials['Liab']
    X5 = financials['Sales'] / financials['TA']
    Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    return Z


def vasicek_wcdr(PD: float, rho: float, X: float) -> float:
    # Worst-Case Default Rate at confidence level X under Vasicek single-factor model.
    from scipy.stats import norm
    return norm.cdf((norm.ppf(PD) + (rho**0.5)*norm.ppf(X)) / (1 - rho)**0.5)


def cva(exposures: list, q: list, R: float, df: float) -> pd.DataFrame:
    # Compute per-year and total CVA with flat annual discounting df^i.
    rows = []
    total = 0.0
    for i, (V, qi) in enumerate(zip(exposures, q), start=1):
        DF_i = df**i
        V_pv = V * DF_i
        LGD = (1 - R)
        CVA_i = LGD * qi * V_pv
        total += CVA_i
        rows.append({'Year': i, 'Exposure': V, 'DF_i': DF_i, 'Exposure_PV': V_pv,
                     'q_i': qi, 'LGD': LGD, 'CVA_i': CVA_i})
    import numpy as np
    import pandas as pd
    df_out = pd.DataFrame(rows)
    df_out.loc[len(df_out)] = {'Year': 'Total', 'Exposure': np.nan, 'DF_i': np.nan,
                               'Exposure_PV': df_out['Exposure_PV'].sum(),
                               'q_i': np.nan, 'LGD': (1-R), 'CVA_i': total}
    return df_out

print('Helper functions loaded')


Helper functions loaded


## Q1 — Market-Implied Credit Risk (CDS)
Worked example: For BBB, spread = 240 bps, recovery = 40% -> hazard rate ~ 0.024 / 0.60 = 0.04 (4%).

Your tasks:
1. Compute hazard rates for BBB and AAA using hazard_rate.
2. Briefly reflect: Why can risk-neutral PD be much higher than historical PD?

In [4]:

# Compute hazard rates
bbb_pd_rn = hazard_rate(240, 0.40)
aaa_pd_rn = hazard_rate(30, 0.40)
print(f'BBB risk-neutral PD ~ {bbb_pd_rn:.4%}')
print(f'AAA risk-neutral PD ~ {aaa_pd_rn:.4%}')

# Auto-check (tolerance +-0.05%)
expected_bbb = 0.04
print('Check BBB:', 'PASS' if abs(bbb_pd_rn - expected_bbb) < 0.0005 else 'REVIEW')


BBB risk-neutral PD ~ 4.0000%
AAA risk-neutral PD ~ 0.5000%
Check BBB: PASS



**Reflection:**
- List two reasons the market-implied PD can exceed rating-agency historical PD.
- How would this affect pricing and risk limits?


## Q2 — Structural Credit Risk (Altman Z)
Use the provided financials to compute Z.
Thresholds: Z > 3.0 (Safe), 1.8 < Z < 3.0 (Alert), Z < 1.8 (Distress).


In [6]:

# Prepare inputs and compute Z
fin = {
    'WC': 50,
    'RE': 20,
    'EBIT': 15,
    'MVE': 80,
    'Liab': 120,
    'Sales': 200,
    'TA': 250
}
Z = altman_z(fin)
print(f'Altman Z-Score = {Z:.3f}')

# Zone classification
zone = 'Safe' if Z > 3.0 else ('Alert' if 1.8 < Z < 3.0 else 'Distress')
print('Zone:', zone)

# Auto-check around expected 1.752
print('Check Z:', 'PASS' if abs(Z - 1.752) < 0.001 else 'REVIEW')


Altman Z-Score = 1.750
Zone: Distress
Check Z: REVIEW


**Reflection:**
- Would you continue funding this originator? Under what covenants?
- What qualitative factors (e.g., underwriting standards) are missing from Z?


## Q3 — Portfolio Loss Modeling (Credit VaR)
Parameters: PD = 0.17% (0.0017), rho = 0.15, confidence X = 99.9%.


In [8]:

PD = 0.0017
X = 0.999
WCDR = vasicek_wcdr(PD, rho, X)
print(f'Worst-Case Default Rate (WCDR) ~ {WCDR:.2%}')

# Portfolio-level example
portfolio_size = 1000
expected_defaults = WCDR * portfolio_size
print(f'Expected defaults at 99.9% ~ {expected_defaults:.1f} out of {portfolio_size}')

# Auto-check ~3% and ~30 defaults
print('Check WCDR:', 'PASS' if abs(WCDR - 0.03) < 0.005 else 'REVIEW')


Worst-Case Default Rate (WCDR) ~ 3.01%
Expected defaults at 99.9% ~ 30.1 out of 1000
Check WCDR: PASS


**Reflection:**
- Why does correlation create fat tails in credit losses?
- How would your VaR change under a buy-and-hold vs. constant-risk strategy?


## Q4 — Counterparty Risk (CVA)
Inputs: Exposure by year = [2M, 3M, 4M]; q = [0.5%, 1.0%, 1.5%]; Recovery R = 40%; Discount 0.95 per year.


In [10]:

exposures = [2_000_000, 3_000_000, 4_000_000]
q = [0.005, 0.010, 0.015]
R = 0.40

cva_df = cva(exposures, q, R, discount_factor_annual)
print(cva_df)
print(f'Total CVA ~ ${cva_df.loc[cva_df["Year"]=="Total","CVA_i"].values[0]:,.0f}')

# Auto-check around ~ $52,797
print('Check CVA:', 'PASS' if abs(cva_df.loc[cva_df['Year']=='Total','CVA_i'].values[0] - 52797) < 1500 else 'REVIEW')


    Year       Exposure   DF_i    Exposure_PV    q_i    LGD       CVA_i
0      1 2,000,000.0000 0.9500 1,900,000.0000 0.0050 0.6000  5,700.0000
1      2 3,000,000.0000 0.9025 2,707,500.0000 0.0100 0.6000 16,245.0000
2      3 4,000,000.0000 0.8574 3,429,500.0000 0.0150 0.6000 30,865.5000
3  Total            NaN    NaN 8,037,000.0000    NaN 0.6000 52,810.5000
Total CVA ~ $52,810
Check CVA: PASS



**Reflection:**
- How would a collateral CSA with a 10-day cure period change your exposure at default?
- What drivers make CVA volatile over time (the CVA Greeks)?


## Q5 — Stress Event: Wrong-Way Risk (AIG Downgrade)
Event: AIG has been downgraded below AA. Collateral calls are triggered.

Assume a 10-day cure period and the following at default:
- Collateral held at t-10: $45M

- Portfolio value at t: $50M

Wrong-way risk regulatory multiplier alpha = 1.4 (apply to CVA as a simple proxy).


In [12]:

V_t = 50_000_000
Collateral_t_minus_10 = 45_000_000
exposure_at_default = max(V_t - Collateral_t_minus_10, 0)
print(f'Exposure at default (EAD) = ${exposure_at_default:,.0f}')

# Re-estimate CVA under stress: amplify default probabilities by 40% (alpha proxy for WWR)
q_stress = [qi * 1.4 for qi in q]
cva_stress_df = cva(exposures, q_stress, R, discount_factor_annual)
CVA_base = cva_df.loc[cva_df['Year']=='Total','CVA_i'].values[0]
CVA_stress = cva_stress_df.loc[cva_stress_df['Year']=='Total','CVA_i'].values[0]
print(f'Base CVA ~ ${CVA_base:,.0f} | Stress CVA (alpha=1.4) ~ ${CVA_stress:,.0f}')

# Auto-check exposure
print('Check EAD:', 'PASS' if exposure_at_default == 5_000_000 else 'REVIEW')


Exposure at default (EAD) = $5,000,000
Base CVA ~ $52,810 | Stress CVA (alpha=1.4) ~ $73,935
Check EAD: PASS



**Reflection:**
- How do downgrade triggers convert a credit deterioration into a liquidity crisis?
- Which hedges or structural changes would mitigate wrong-way risk in practice?
